# Notebook 05 — Data Quality Fix

1. Re-chunk the PDF respecting **sentence boundaries** (`src/chunker_v2.py`).
2. Generate training queries with **an LLM** instead of extracting them
   (`src/query_generator.py`).
3. Build new train/val splits and save them.

## Setup

In [1]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/NLP PROJECT/Neural_Search_Engine-main'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q pymupdf anthropic openai
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

print('Working directory:', os.getcwd())

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 46.6 MB/s eta 0:00:00
Working directory: /content/drive/MyDrive/NLP PROJECT/Neural_Search_Engine-main


---
## Step 1 — Re-chunk the PDF with sentence boundaries

In [ ]:
import json
from src.utils import extract_clean_text_from_pdf
from src.chunker_v2 import SentenceAwareChunker, split_into_sentences

PDF_PATH = 'data/jurafsky_martin.pdf'
OUTPUT_PATH = 'data/processed/jurafsky_chunks_v2.json'

# Step 1a: extract clean text from the PDF (reuses your existing utility)
raw_text = extract_clean_text_from_pdf(PDF_PATH)
print(f'Extracted {len(raw_text):,} characters')

# Step 1b: chunk with sentence-aware logic
chunker = SentenceAwareChunker(target_words=220, max_words=300, overlap_sentences=2)
chunks_v2 = chunker.chunk(raw_text)

# Step 1c: save
os.makedirs('data/processed', exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(chunks_v2, f, indent=2, ensure_ascii=False)

print(f'\nWrote {len(chunks_v2)} chunks -> {OUTPUT_PATH}')
wcs = [c['word_count'] for c in chunks_v2]
print(f'Word count: min={min(wcs)}, max={max(wcs)}, avg={sum(wcs)/len(wcs):.0f}')

Reading PDF from: data/jurafsky_martin.pdf
Extracted 520 pages of clean text.
Extracted 1,393,036 characters

Wrote 1410 chunks -> data/processed/jurafsky_chunks_v2.json
Word count: min=101, max=414, avg=239


In [2]:
import json
OUTPUT_PATH = 'data/wiki/wiki_chunks.json'
# 1. Load the json data from the file
with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

# 2. Find the maximum word_count
max_words = max(item['word_count'] for item in chunks)

print(f"The maximum word count is: {max_words}")

The maximum word count is: 959


In [ ]:
# Verify the fix — every chunk should now start with a real sentence
print('First sentence of first 5 chunks (compare to your old jurafsky_chunks.json):')
for c in chunks_v2[:5]:
    first = split_into_sentences(c['content'])[0]
    safe = first.encode('ascii', 'replace').decode()
    print(f"  [{c['id']}] {safe[:140]}")

First sentence of first 5 chunks (compare to your old jurafsky_chunks.json):
  [chunk_0001] WORDS AND TOKENS speci?c language, at a speci?c time, in a speci?c place, for a speci?c function.
  [chunk_0002] Thus, for example, if we?re processing text that uses features of African American English (AAE) or African AAE American Vernacular English 
  [chunk_0003] Code switching is enormously code switching common across the world; here are examples showing Spanish and (transliterated) Hindi code switc
  [chunk_0004] Language changes over time, and for some languages we have good corpora of texts from different historical periods.
  [chunk_0005] Annotation process: What are the annotations, what are the demographics of the annotators, how were they trained, how was the data annotated


Generating queries for chunks using LLM's

Two passes:
1. cell 1 → writes prompt files to `data/prompts/`. Then respondes are saved as
   `data/prompts/response_NNN.json`.
2. cell 2 → ingests the responses into `data/processed/llm_pairs.json`.

In [ ]:
from src.query_generator import ManualBatchGenerator
manual = ManualBatchGenerator(n_queries=2)
manual.batch_prompts(chunks_v2, out_dir='data/prompts', batch_size=20)

Wrote 71 prompt files to data/prompts/
Paste each prompt into a chat UI. Save responses as response_NNN.json.


71

In [ ]:
manual.ingest_responses(
    responses_dir='data/prompts',
    chunks=chunks_v2,
    output_path='data/processed/llm_pairs.json',
)

  [skip] response_006.json: bad JSON (Expecting property name enclosed in double quotes: line 47 column 1 (char 2003))
  [skip] response_007.json: bad JSON (Expecting value: line 95 column 13 (char 4037))
  [skip] queries not in corpus
  [skip] response_048.json: bad JSON (Expecting ',' delimiter: line 130 column 16 (char 4297))
  [skip] queries not in corpus
  [skip] chunk_1440 not in corpus
Wrote 2580 pairs -> data/processed/llm_pairs.json


[{'query': 'How many distinct languages are cataloged in the online Ethnologue directory according to the text?',
  'positive_id': 'chunk_0001',
  'positive_text': 'WORDS AND TOKENS speciﬁc language, at a speciﬁc time, in a speciﬁc place, for a speciﬁc function. Perhaps the most important dimension of variation is the language. NLP algorithms are most useful when they apply across many languages. The world has 7097 languages at the time of this writing, according to the online Ethnologue catalog (Simons and Fennig, 2018). It is important to test algorithms on more than one language, and particularly on languages with different properties; by contrast there is an unfortunate current tendency for NLP algorithms to be developed or tested just on English (Bender, 2019). Even when algorithms are developed beyond English, they tend to be developed for the ofﬁcial languages of large industrialized nations (Chinese, Spanish, Japanese, German etc.), but we don’t want to limit tools to just thes

In [ ]:
with open('data/processed/llm_pairs.json', encoding='utf-8') as f:
    llm_pairs = json.load(f)

print(f'Loaded {len(llm_pairs)} (query, positive) pairs')
print(f'Unique chunks covered: {len(set(p["positive_id"] for p in llm_pairs))}')
print(f'Avg queries per chunk: {len(llm_pairs) / len(set(p["positive_id"] for p in llm_pairs)):.1f}')

print('\n--- Sample LLM-generated queries (compare to first-sentence garbage) ---\n')
for p in llm_pairs[::200][:5]:
    print(f"Q: {p['query']}")
    print(f"   matches [{p['positive_id']}]: {p['positive_text'][:120]}...\n")

Loaded 2580 (query, positive) pairs
Unique chunks covered: 1270
Avg queries per chunk: 2.0

--- Sample LLM-generated queries (compare to first-sentence garbage) ---

Q: How many distinct languages are cataloged in the online Ethnologue directory according to the text?
   matches [chunk_0001]: WORDS AND TOKENS speciﬁc language, at a speciﬁc time, in a speciﬁc place, for a speciﬁc function. Perhaps the most impor...

Q: What specific mathematical conversion replaces individual elements to establish a zero mean and unit standard deviation?
   matches [chunk_0141]: Scaling input features: When different input features have extremely different ranges of values, it’s common to rescale ...

Q: What role does the hyperparameter k perform in regulating dataset proportions for skip-gram with negative sampling?
   matches [chunk_0260]: 5.5.2 Learning skip-gram embeddings The learning algorithm for skip-gram embeddings takes as input a corpus of text, and...

Q: What name is assigned to language m

---
 Add random negatives and write train/val pairs

In [ ]:
import random
from src.query_generator import add_random_negatives
from src.dataset import save_pairs

# 4a: enrich each (query, positive) pair with a random distant negative
triplets = add_random_negatives(llm_pairs, chunks_v2, min_distance=20, seed=42)
print(f'Enriched to {len(triplets)} (query, positive, negative) triplets')

# 4b: shuffle and split 90/10 train/val
rng = random.Random(42)
rng.shuffle(triplets)
cut = int(len(triplets) * 0.9)
train_pairs = triplets[:cut]
val_pairs   = triplets[cut:]

# 4c: save with the SAME filenames that 02_training.ipynb expects
save_pairs(train_pairs, 'data/processed/train_pairs.json')
save_pairs(val_pairs,   'data/processed/val_pairs.json')

print(f'\nTrain pairs: {len(train_pairs)}')
print(f'Val pairs:   {len(val_pairs)}')
print(f'Test queries (unchanged): 10 in data/evaluation_set.csv')

Enriched to 2580 (query, positive, negative) triplets
Saved 2322 pairs -> data/processed/train_pairs.json
Saved 258 pairs -> data/processed/val_pairs.json

Train pairs: 2322
Val pairs:   258
Test queries (unchanged): 10 in data/evaluation_set.csv


In [ ]:
import shutil

if os.path.exists('data/processed/jurafsky_chunks.json'):
    shutil.copy('data/processed/jurafsky_chunks.json', 'data/processed/jurafsky_chunks_v1_backup.json')
    print('Backed up old chunks to jurafsky_chunks_v1_backup.json')

shutil.copy('data/processed/jurafsky_chunks_v2.json', 'data/processed/jurafsky_chunks.json')
print('Promoted v2 chunks to the canonical jurafsky_chunks.json')

with open('data/processed/jurafsky_chunks.json', encoding='utf-8') as f:
    final = json.load(f)
print(f'Canonical corpus now has {len(final)} chunks.')

Backed up old chunks to jurafsky_chunks_v1_backup.json
Promoted v2 chunks to the canonical jurafsky_chunks.json
Canonical corpus now has 1410 chunks.
